In [1]:
import numpy as np
import networkx as nx


%load_ext autoreload
%autoreload 2

In [2]:
graphs_er = np.load("../Tutorials/ER/resolution-1/graphs-1_p-0.4.npy", allow_pickle=True)
graphs_er = [g[0] for g in graphs_er]
graphs_er_110_160 = np.load("graphs/erdos-renyi_110-160/erdos-renyi_110-160.npy", allow_pickle=True)
g_er_169 = np.load("graphs/erdos-renyi_110-160/erdos-renyi_169.npy", allow_pickle=True)
graphs_er = graphs_er + graphs_er_110_160.tolist() + g_er_169.tolist()

graphs_pwl = np.load("../Tutorials/Power-law/resolution-1/graphs-1_m-1_p-0.1.npy", allow_pickle=True)
graphs_pwl = [g[0] for g in graphs_pwl]
graphs_pwl_110_160 = np.load("graphs/powerlaw_110-160/powerlaw_graphs_110-160.npy", allow_pickle=True)
g_pwl_169 = np.load("graphs/powerlaw_110-160/powerlaw_graph_169.npy", allow_pickle=True)
graphs_pwl = graphs_pwl + graphs_pwl_110_160.tolist() + g_pwl_169.tolist()

In [3]:
results_dir_er = f"ER/resolution-1/graphs-1_p-0.4"
results_dir_pwl = f"Power-law/resolution-1/graphs-1_m-1_p-0.1"

In [8]:
import os

res_path = f"{results_dir_pwl}/graph_size=169-extra-10_2/"
os.makedirs(res_path, exist_ok=True)

In [9]:
from Qommunity.samplers.hierarchical.advantage_sampler import AdvantageSampler
from Qommunity.iterative_searcher import IterativeSearcher

G = graphs_pwl[16]
assert G.number_of_nodes() == 169

adv = AdvantageSampler(G, num_reads=100, 
                       version="Advantage_system6.4", region="na-west-1", 
                       use_clique_embedding=True,
                       elapse_times=True,
                       return_metadata=True)

it_searcher = IterativeSearcher(adv)

In [10]:
res = it_searcher.run_with_sampleset_info(num_runs=10, saving_path=res_path, return_metadata=True)

100%|██████████| 10/10 [00:00<00:00, 4981.95it/s]


In [9]:
# original runs
from sampleset_data.iterative_utils import IterativeSearchGraphResults
from searchers.utils import HierarchicalRunMetadata


num_runs = 20
path = res_path
hierarchical_metadatas = []

for iter in range(num_runs):
    base_filename = f"{path}/_iter_{iter}"
    hm = HierarchicalRunMetadata.load_from_files(base_filename)
    hierarchical_metadatas.append(hm)

communities = np.load(f"{path}/_communities.npy", allow_pickle=True)
modularities = np.load(f"{path}/_modularities.npy", allow_pickle=True)
times = np.load(f"{path}/_times.npy", allow_pickle=True)
division_modularities = np.load(f"{path}/_division_modularities.npy", allow_pickle=True)
division_trees = np.load(f"{path}/_division_trees.npy", allow_pickle=True)

iterative_graph_results = IterativeSearchGraphResults(
    graph_ref = G,
    graph_size=169,
    communities = communities,
    modularities = modularities,
    times = times,
    division_modularities = division_modularities,
    division_trees = division_trees,
    hierarchical_metadatas=hierarchical_metadatas
)

In [ ]:
for run_idx in range(20):
    division_tree = division_trees[run_idx]
    sampleset_metadatas = hierarchical_metadatas[run_idx]


    from utils import recover_ordering, recover_info_ordering, plot_tree_extended


    nodes, root = recover_ordering(G=G, division_tree=division_tree)
    nodes_info, root_info = recover_info_ordering(root=root, nodes=nodes, sampleset_metadata=sampleset_metadatas)

    from matplotlib import colormaps


    cmap = colormaps.get_cmap("PuBu")
    plot_tree_extended(root_info, division_modularities[run_idx], cmap=cmap, figsize=(25,13))